# 🪖 Helmet Detection - YOLOv8 Training & NCNN Export Notebook
สมุดบันทึกสำหรับเทรนโมเดลตรวจจับการสวมหมวกกันน็อก (YOLOv8n) บน **Google Colab (GPU T4)**
รองรับชุดข้อมูล **1,108 ภาพ** (`hwlmet.yolov8 (2).zip`) พร้อมส่งออกโมเดลในชื่อ **`helmet_detector`**

---
### 📊 สรุปชุดข้อมูล (1,108 รูปภาพ):
- 🏋️ **Train Set:** 775 รูป (70%)
- 🧪 **Validation Set:** 222 รูป (20%)
- 🎯 **Test Set:** 111 รูป (10%)
- 🏷️ **Classes (2 คลาส):** `['with-helmet', 'without-helmet']`

### ⚡ การตั้งค่าการเทรนที่ปรับจูนให้เหมาะสม (Optimal Hyperparameters):
- **Epochs:** `100` (จำนวนรอบที่พอดีที่สุดสำหรับ 1,100 รูป ไม่น้อยไปและไม่เกิด Overfitting)
- **Patience (Early Stopping):** `25` (หยุดอัตโนมัติหาก mAP ไม่พัฒนาต่อเนื่อง 25 รอบ)
- **Batch Size:** `16` (ให้ Gradient กระจายตัวได้ดี เหมาะกับชุดข้อมูลขนาด 1,000 รูป)
- **Image Size:** `640` (มาตรฐานคมชัดสูงของ YOLOv8)
- **Base Model:** `yolov8n.pt` (รันไว แม่นยำ เหมาะสำหรับแปลงเป็น NCNN ลงบอร์ด/PC)
- **Export Names:** `helmet_detector.pt` และโฟลเดอร์ `helmet_detector_ncnn/`

---

## ⚡ ขั้นตอนที่ 1: ตรวจสอบการ์ดจอ (GPU)

In [ ]:
# ตรวจสอบว่าเปิดใช้งาน GPU หรือยัง (ควรเป็น Tesla T4 หรือสูงกว่า)
!nvidia-smi

import torch
print("\nCUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("⚠️ คำเตือน: ยังไม่ได้เปิด GPU! กรุณาไปที่เมนู Runtime -> Change runtime type -> เลือก T4 GPU")

## 📦 ขั้นตอนที่ 2: ติดตั้ง Ultralytics (YOLOv8)

In [ ]:
!pip install -q ultralytics

import ultralytics
print(f"✅ ติดตั้ง Ultralytics เรียบร้อยแล้ว: v{ultralytics.__version__}")

## 📁 ขั้นตอนที่ 3: เตรียม Dataset (`hwlmet.yolov8 (2).zip`)

💡 **วิธีนำเข้า Dataset:**
1. ลากไฟล์ `hwlmet.yolov8 (2).zip` (ขนาด ~430 MB) โยนใส่แถบไฟล์ด้านซ้ายมือของ Colab (`/content/`)
2. หรือเมานต์ผ่าน Google Drive ถ้าคุณอัปโหลดเก็บไว้ในไดรฟ์
3. รันโค้ดด้านล่าง ระบบจะค้นหาและแตกไฟล์อัตโนมัติ

In [ ]:
import os
import glob
import shutil
import zipfile

extract_dir = "/content/dataset"
os.makedirs(extract_dir, exist_ok=True)

# 1. ค้นหาไฟล์ zip ของ dataset อัตโนมัติ (รองรับ hwlmet.yolov8 (2).zip, dataset.zip หรือชื่ออื่นๆ)
all_zips = [f for f in glob.glob('/content/*.zip') if 'bundle' not in os.path.basename(f) and 'helmet_detector' not in os.path.basename(f) and 'best' not in os.path.basename(f)]

target_zip = None
for z in all_zips:
    bname = os.path.basename(z).lower()
    if 'hwlmet' in bname or 'helmet' in bname or 'dataset' in bname or '(2)' in bname:
        target_zip = z
        break

if not target_zip and all_zips:
    target_zip = all_zips[0]

# หากมีไฟล์ dataset.part_* ให้รวมไฟล์อัตโนมัติ
part_files = sorted([f for f in os.listdir('/content') if f.startswith('dataset.part_')])
if part_files and not target_zip:
    combined_zip = "/content/dataset.zip"
    print(f"กำลังรวมไฟล์ {len(part_files)} พาร์ท เข้าเป็น {combined_zip}...")
    with open(combined_zip, 'wb') as outfile:
        for pf in part_files:
            p_path = os.path.join('/content', pf)
            with open(p_path, 'rb') as infile:
                outfile.write(infile.read())
    target_zip = combined_zip

if target_zip and os.path.exists(target_zip):
    print(f"📦 กำลังแตกไฟล์ Dataset: {target_zip} ({os.path.getsize(target_zip)/(1024*1024):.2f} MB)...")
    with zipfile.ZipFile(target_zip, 'r') as zip_ref:
        for member in zip_ref.namelist():
            normalized = member.replace('\\', '/')
            target_path = os.path.join(extract_dir, normalized)
            if normalized.endswith('/'):
                os.makedirs(target_path, exist_ok=True)
            else:
                os.makedirs(os.path.dirname(target_path), exist_ok=True)
                with zip_ref.open(member) as src, open(target_path, 'wb') as dst:
                    dst.write(src.read())
    print("✅ แตกไฟล์ Dataset สำเร็จเรียบร้อย!")
else:
    print("⚠️ ยังไม่พบไฟล์ Dataset (.zip) ใน /content/")
    print("👉 กรุณาลากไฟล์ 'hwlmet.yolov8 (2).zip' หรือ 'dataset.zip' มาวางในช่อง Files ด้านซ้าย แล้วรัน Cell นี้ใหม่อีกครั้ง!")

## 🔍 ขั้นตอนที่ 4: ตรวจสอบความถูกต้องของ Dataset และ `data.yaml`

In [ ]:
yaml_path = "/content/dataset/data.yaml"

# เขียนทับ data.yaml ด้วย Absolute Path บน Colab เพื่อป้องกัน Path เพี้ยน 100%
yaml_content = """path: /content/dataset
train: train/images
val: valid/images
test: test/images

nc: 2
names: ['with-helmet', 'without-helmet']
"""

with open(yaml_path, "w", encoding="utf-8") as f:
    f.write(yaml_content.strip())

train_img_dir = "/content/dataset/train/images"
valid_img_dir = "/content/dataset/valid/images"
test_img_dir = "/content/dataset/test/images"

n_train = len([f for f in os.listdir(train_img_dir) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]) if os.path.exists(train_img_dir) else 0
n_val = len([f for f in os.listdir(valid_img_dir) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]) if os.path.exists(valid_img_dir) else 0
n_test = len([f for f in os.listdir(test_img_dir) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]) if os.path.exists(test_img_dir) else 0
total = n_train + n_val + n_test

print("=" * 55)
print("📊 สรุปจำนวนรูปภาพใน Dataset ที่โหลดสำเร็จ:")
print(f"  • รูปภาพสำหรับ Train      : {n_train} รูป")
print(f"  • รูปภาพสำหรับ Validation : {n_val} รูป")
print(f"  • รูปภาพสำหรับ Test       : {n_test} รูป")
print(f"  • รวมทั้งหมด             : {total} รูป")
print("=" * 55)
with open(yaml_path, "r", encoding="utf-8") as f:
    print("เนื้อหา data.yaml:\n" + f.read())
print("=" * 55)

## 🎯 ขั้นตอนที่ 5: เริ่มเทรนโมเดล YOLOv8n (ตั้งค่าเหมาะสมสำหรับ 1,108 รูป)

⚙️ **การปรับแต่งที่เหมาะสม:**
- `epochs=100`: จำนวนรอบที่เหมาะสมที่สุดสำหรับชุดข้อมูลระดับ 1,100 รูป (ประมาณ 4,800 gradient steps)
- `patience=25`: Early Stopping หยุดอัตโนมัติหากผลประเมินไม่ดีขึ้นติดต่อกัน 25 รอบ (ประหยัดเวลา ไม่ Overfit)
- `batch=16`: ปรับสมดุลระหว่างความเร็วและการคำนวณ Gradient บน T4 GPU
- `imgsz=640`: ความละเอียดมาตรฐานสูง คมชัด แยกแยะหมวกได้แม่นยำ

In [ ]:
import torch
from ultralytics import YOLO

# โหลดโมเดลตั้งต้น YOLOv8n Pre-trained
model = YOLO('yolov8n.pt')

print("🚀 กำลังเริ่มการเทรนโมเดล...")
results = model.train(
    data='/content/dataset/data.yaml',
    epochs=100,          # 100 Epochs เหมาะสมที่สุดสำหรับ 1,108 รูป
    patience=25,         # Early stopping หยุดอัตโนมัติถ้านิ่งเกิน 25 epochs
    batch=16,            # Batch size ขนาด 16 เหมาะกับ dataset ขนาดนี้
    imgsz=640,           # ขนาดภาพ 640x640
    device=0 if torch.cuda.is_available() else 'cpu',
    project='/content/runs/detect',
    name='train',
    exist_ok=True,
    verbose=True,
    save=True,
    plots=True           # บันทึกกราฟ Confusion Matrix, PR curve, Results อัตโนมัติ
)

print("\n🎉 เทรนโมเดลเสร็จสมบูรณ์เรียบร้อยแล้ว!")

## 📊 ขั้นตอนที่ 6: แสดงผลกราฟความแม่นยำ (Results & Confusion Matrix)

In [ ]:
from IPython.display import Image, display

results_png = "/content/runs/detect/train/results.png"
cm_png = "/content/runs/detect/train/confusion_matrix.png"

if os.path.exists(results_png):
    print("📈 กราฟผลการเทรน (Loss & mAP50 / mAP50-95):")
    display(Image(results_png))

if os.path.exists(cm_png):
    print("🎯 Confusion Matrix (ความแม่นยำรายคลาส with-helmet / without-helmet):")
    display(Image(cm_png))

## 📦 ขั้นตอนที่ 7: Export NCNN & รวบรวมโมเดล `helmet_detector` ลงในโฟลเดอร์เดียวกัน
ระบบจะบันทึกโมเดลในชื่อ **`helmet_detector`** และรวมทุกอย่างไว้ในโฟลเดอร์ `helmet_trained_bundle/`:
- 📄 `helmet_detector.pt` (โมเดล PyTorch สำหรับความแม่นยำสูงสุด)
- 📁 โฟลเดอร์ `helmet_detector_ncnn/` (โมเดล NCNN สำหรับรันบน PC GUI / ESP32)
- 📈 กราฟสรุปผลทั้งหมด (`results.png`, `confusion_matrix.png`, PR curve ฯลฯ)
- 📑 ตารางสถิติ `results.csv`

จากนั้นจะบีบอัดเป็น **`helmet_trained_bundle.zip`** ไฟล์เดียวจบ ไม่กระจัดกระจาย!

In [ ]:
import os
import shutil
import glob
from ultralytics import YOLO

raw_best = "/content/runs/detect/train/weights/best.pt"
train_dir = "/content/runs/detect/train"

# 1. สร้างโฟลเดอร์หลักสำหรับรวมทุกอย่างไว้ที่เดียว
bundle_dir = "/content/helmet_trained_bundle"
if os.path.exists(bundle_dir):
    shutil.rmtree(bundle_dir)
os.makedirs(bundle_dir, exist_ok=True)

# 2. บันทึกและเปลี่ยนชื่อเป็น helmet_detector.pt
target_pt = os.path.join(bundle_dir, "helmet_detector.pt")
shutil.copy(raw_best, target_pt)
shutil.copy(raw_best, "/content/helmet_detector.pt")
last_weights = os.path.join(train_dir, "weights", "last.pt")
if os.path.exists(last_weights):
    shutil.copy(last_weights, os.path.join(bundle_dir, "last.pt"))
print(f"✅ บันทึกโมเดล: helmet_detector.pt ({os.path.getsize(target_pt)/(1024*1024):.2f} MB)")

# 3. Export เป็น NCNN Format ในชื่อ helmet_detector_ncnn
print("⚙️ กำลังแปลงโมเดลเป็น NCNN Format...")
trained_model = YOLO(raw_best)
ncnn_exported = trained_model.export(format="ncnn", imgsz=640)

# ย้ายและตั้งชื่อโฟลเดอร์ NCNN ให้เป็น helmet_detector_ncnn ภายใน bundle_dir
target_ncnn_dir = os.path.join(bundle_dir, "helmet_detector_ncnn")
shutil.copytree(ncnn_exported, target_ncnn_dir, dirs_exist_ok=True)
print(f"✅ บันทึกโฟลเดอร์ NCNN: helmet_detector_ncnn/")

# 4. คัดลอกกราฟและตารางผลลัพธ์ทั้งหมด
for img_file in glob.glob(os.path.join(train_dir, "*.png")):
    shutil.copy(img_file, bundle_dir)
for csv_file in glob.glob(os.path.join(train_dir, "*.csv")):
    shutil.copy(csv_file, bundle_dir)
print(f"✅ คัดลอกกราฟและไฟล์สถิติทั้งหมดเข้าโฟลเดอร์สำเร็จ")

# 5. บีบอัดเป็น ZIP ก้อนเดียวจบ
bundle_zip_base = "/content/helmet_trained_bundle"
zip_out = shutil.make_archive(bundle_zip_base, 'zip', bundle_dir)
print("=" * 60)
print(f"🎉 มัดรวมทุกไฟล์ลงใน ZIP เดียวสำเร็จ:")
print(f"   -> {zip_out} ({os.path.getsize(zip_out)/(1024*1024):.2f} MB)")
print("=" * 60)

## 💾 ขั้นตอนที่ 8: ดาวน์โหลด ZIP มัดรวมไฟล์เดียวลงเครื่องคอมพิวเตอร์

In [ ]:
from google.colab import files

bundle_zip = "/content/helmet_trained_bundle.zip"
print(f"🚀 กำลังเริ่มดาวน์โหลด '{os.path.basename(bundle_zip)}' ไฟล์เดียวจบ...")
try:
    files.download(bundle_zip)
    print("✅ เบราว์เซอร์เริ่มดาวน์โหลดไฟล์แล้ว! พอโหลดเสร็จ นำไปแตกไฟล์ลงโฟลเดอร์โปรเจกต์ได้เลย")
except Exception as e:
    print(f"⚠️ เกิดข้อผิดพลาดในการดาวน์โหลดอัตโนมัติ: {e}")
    print(f"👉 คุณสามารถคลิกขวาที่ไฟล์ '{os.path.basename(bundle_zip)}' ในแถบ Files ทางซ้ายมือ แล้วกด Download เองได้เลย!")